# 🚀 Bladder Tissue Classification - WITH Augmentation

## LOPO Cross-Validation Pipeline
- **Lab Normalization** → Multi-Scale Patches → **Data Augmentation** → Dual Backbone → CLAM
- **WITH Data Augmentation** - Better Generalization
- Class-weighted loss with early stopping
- Test-Time Augmentation (TTA) enabled

---

In [ ]:
# ============================================================
# CELL 1: SETUP
# ============================================================
import os, re, gc, copy, time, random, warnings, json
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
import cv2
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import torchvision.transforms as T

from sklearn.metrics import (
    classification_report, confusion_matrix,
    balanced_accuracy_score, accuracy_score
)
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

SEED = 42
def seed_everything(s=SEED):
    random.seed(s); np.random.seed(s)
    os.environ['PYTHONHASHSEED'] = str(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything()
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

In [ ]:
# ============================================================
# CELL 2: CONFIGURATION
# ============================================================
class CFG:
    data_root = "/kaggle/input/endoscopic-bladder-tissue-classification-dataset/EndoscopicBladderTissue"

    class_names = ['HGC', 'LGC', 'Normal']  # Combined NST + NTL into Normal
    num_classes = 3

    # Preprocessing
    image_resize = 512
    clahe_clip = 1.5
    clahe_grid = (16, 16)

    # Multi-scale patches
    patch_scales = [96, 128, 192]
    patch_output_size = 224
    patch_stride_frac = 0.5
    min_tissue = 0.45
    max_bright = 245
    min_bright = 15
    min_sat = 10
    min_focus = 8.0
    top_quality_frac = 0.75

    # Limit patches per image to control memory and speed
    max_patches_per_image = 80

    # Feature extraction
    feat_batch = 32
    feat_dim = 1024  # updated after backbone load

    # CLAM
    mil_hidden = 512
    mil_dropout = 0.20
    clam_k_sample = 12
    feat_noise_std = 0.03
    feat_drop_p = 0.1
    inst_loss_w = 0.3
    bag_loss_w = 1.0

    # Training
    epochs = 100
    patience = 25
    lr = 1e-4
    wd = 5e-5
    grad_clip = 1.0
    max_patches_train = 400
    max_patches_test = 600

    # Augmentation
    n_augments = 2

    # Ensemble
    n_ensemble = 5
    
    # Test-Time Augmentation
    use_tta = True
    tta_rounds = 5

    # Class weights — computed from data
    class_weights = None

IMNET_MEAN = torch.tensor([0.485,0.456,0.406]).view(1,3,1,1).to(DEVICE)
IMNET_STD  = torch.tensor([0.229,0.224,0.225]).view(1,3,1,1).to(DEVICE)

print("✓ Config loaded - WITH AUGMENTATION")

In [ ]:
# ============================================================
# CELL 3: DATASET LOADING + CLASS WEIGHTS
# ============================================================
records = []
pattern = re.compile(r'pt[_]?0*(\d+)')

# Map original labels to combined labels
label_mapping = {
    'HGC': 'HGC',
    'LGC': 'LGC',
    'NST': 'Normal',
    'NTL': 'Normal'
}

for label in os.listdir(CFG.data_root):
    class_path = os.path.join(CFG.data_root, label)
    if not os.path.isdir(class_path):
        continue
    if label not in label_mapping:
        print(f"  ⚠ Skipping unknown folder: {label}")
        continue
    for img_name in os.listdir(class_path):
        match = pattern.search(img_name)
        if match:
            patient_id = int(match.group(1))
            records.append({
                "path": os.path.join(class_path, img_name),
                "label": label_mapping[label],
                "original_label": label,
                "patient": patient_id,
                "filename": img_name
            })

df = pd.DataFrame(records)
class_to_idx = {c: i for i, c in enumerate(CFG.class_names)}
idx_to_class = {i: c for c, i in class_to_idx.items()}
df["target"] = df["label"].map(class_to_idx)

print(f"Total images: {len(df)}")
print(f"Total patients: {df.patient.nunique()}")
print(f"Classes: {CFG.class_names}")
print(f"Mapping: {class_to_idx}")

# ---- Compute class weights ----
class_counts = df['label'].value_counts()
total = len(df)
weights = []
print(f"\nClass distribution and weights:")
for cls in CFG.class_names:
    count = class_counts.get(cls, 0)
    w = total / (CFG.num_classes * max(count, 1))
    weights.append(w)
    print(f"  {cls}: {count} images ({count/total*100:.1f}%) → weight={w:.3f}")

CFG.class_weights = torch.tensor(weights, dtype=torch.float32).to(DEVICE)
print(f"\nClass weights tensor: {CFG.class_weights}")

# ---- Patient summary ----
PATIENTS = sorted(df.patient.unique())
N_PATIENTS = len(PATIENTS)

print(f"\n{'Patient':<10} {'#Imgs':<8} {'Distribution'}")
print("-" * 65)
for pid in PATIENTS:
    pdf = df[df.patient == pid]
    counts = Counter(pdf.label.values)
    dist = ", ".join(f"{k}:{v}" for k, v in sorted(counts.items()))
    print(f"  P{pid:<8} {len(pdf):<8} {dist}")

print(f"\n✓ {N_PATIENTS} patients, {len(df)} images, {CFG.num_classes} classes")

In [ ]:
# ============================================================
# CELL 4: LAB NORMALIZATION + CLAHE
# ============================================================
class LabNormalizer:
    def __init__(self):
        self.ref = None

    def fit(self, images_bgr):
        stats = {'L': [], 'a': [], 'b': []}
        for img in images_bgr:
            lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB).astype(np.float32)
            for i, ch in enumerate(['L', 'a', 'b']):
                stats[ch].append({
                    'm': lab[:,:,i].mean(),
                    's': lab[:,:,i].std() + 1e-6
                })
        self.ref = {
            ch: {'m': np.median([s['m'] for s in stats[ch]]),
                 's': np.median([s['s'] for s in stats[ch]])}
            for ch in ['L', 'a', 'b']
        }
        return self

    def transform(self, img_bgr):
        lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB).astype(np.float32)
        for i, ch in enumerate(['L', 'a', 'b']):
            c = lab[:,:,i]
            sm, ss = c.mean(), c.std() + 1e-6
            lab[:,:,i] = np.clip(
                (c - sm) * (self.ref[ch]['s'] / ss) + self.ref[ch]['m'], 0, 255
            )
        lab = lab.astype(np.uint8)
        clahe = cv2.createCLAHE(clipLimit=CFG.clahe_clip, tileGridSize=CFG.clahe_grid)
        lab[:,:,0] = clahe.apply(lab[:,:,0])
        return cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)


def load_image(path, norm=None):
    img = cv2.imread(path)
    if img is None:
        raise FileNotFoundError(path)
    h, w = img.shape[:2]
    s = CFG.image_resize / max(h, w)
    if s != 1:
        img = cv2.resize(img, (int(w*s), int(h*s)), interpolation=cv2.INTER_AREA)
    if norm:
        img = norm.transform(img)
    return img


def fit_normalizer_excluding(exclude_pid=None):
    """Fit normalizer on all patients EXCEPT exclude_pid."""
    samples = []
    for pid in PATIENTS:
        if pid == exclude_pid:
            continue
        for fp in df[df.patient == pid].path.values[:12]:
            try:
                img = cv2.imread(fp)
                if img is not None:
                    h, w = img.shape[:2]
                    s = CFG.image_resize / max(h, w)
                    if s != 1:
                        img = cv2.resize(img, (int(w*s), int(h*s)))
                    samples.append(img)
            except:
                pass
    return LabNormalizer().fit(samples)


# Fit a global normalizer for feature extraction
global_normalizer = fit_normalizer_excluding(exclude_pid=None)
print(f"✓ Global normalizer fitted")

In [ ]:
# ============================================================
# CELL 5: MULTI-SCALE PATCHES + QUALITY FILTER
# ============================================================

def compute_quality(patch_bgr):
    hsv = cv2.cvtColor(patch_bgr, cv2.COLOR_BGR2HSV)
    v = hsv[:,:,2].astype(float)
    s = hsv[:,:,1].astype(float)

    mask = (v < CFG.max_bright) & (v > CFG.min_bright) & (s > CFG.min_sat)
    tissue_frac = mask.sum() / mask.size
    if tissue_frac < CFG.min_tissue:
        return -1.0

    gray = cv2.cvtColor(patch_bgr, cv2.COLOR_BGR2GRAY)
    focus = cv2.Laplacian(gray, cv2.CV_64F).var()
    if focus < CFG.min_focus:
        return -1.0

    focus_norm = min(focus / 100.0, 1.0)
    sat_std = s[mask].std() / 50.0 if mask.sum() > 10 else 0
    sat_norm = min(sat_std, 1.0)
    edges = cv2.Canny(gray, 50, 150)
    edge_density = min(edges.sum() / (255.0 * edges.size) * 10, 1.0)

    return 0.3*tissue_frac + 0.3*focus_norm + 0.2*sat_norm + 0.2*edge_density


def extract_multiscale_patches(image_bgr, max_patches=None):
    """Extract patches at multiple scales."""
    if max_patches is None:
        max_patches = CFG.max_patches_per_image

    H, W = image_bgr.shape[:2]
    candidates = []

    for scale in CFG.patch_scales:
        if scale > min(H, W):
            continue
        stride = max(1, int(scale * CFG.patch_stride_frac))
        for y in range(0, H - scale + 1, stride):
            for x in range(0, W - scale + 1, stride):
                crop = image_bgr[y:y+scale, x:x+scale]
                q = compute_quality(crop)
                if q > 0:
                    resized = cv2.resize(
                        crop,
                        (CFG.patch_output_size, CFG.patch_output_size),
                        interpolation=cv2.INTER_AREA
                    )
                    candidates.append((resized, q, scale))

    if len(candidates) == 0:
        return [], [], []

    # Sort by quality, keep top fraction
    candidates.sort(key=lambda x: x[1], reverse=True)
    n_keep = max(1, int(len(candidates) * CFG.top_quality_frac))
    candidates = candidates[:n_keep]

    # Cap at max_patches
    if len(candidates) > max_patches:
        candidates = candidates[:max_patches]

    return ([c[0] for c in candidates],
            [c[1] for c in candidates],
            [c[2] for c in candidates])


# Test
test_img = load_image(df.iloc[0].path, global_normalizer)
tp, ts, tsc = extract_multiscale_patches(test_img)
print(f"✓ Test: {test_img.shape} → {len(tp)} patches (cap={CFG.max_patches_per_image})")
if tsc:
    for s, c in sorted(Counter(tsc).items()):
        print(f"    Scale {s}: {c}")
del tp, ts, tsc, test_img; gc.collect()

In [ ]:
# ============================================================
# CELL 6: CYSTOSCOPY AUGMENTATION
# ============================================================

class GaussianNoise:
    def __init__(self, std=0.02):
        self.std = std
    def __call__(self, t):
        return torch.clamp(t + torch.randn_like(t) * self.std, 0, 1)

class CystoAugmentor:
    """Medical image augmentation for cystoscopy."""
    def __init__(self):
        self.transform = T.Compose([
            T.RandomRotation(360),
            T.RandomHorizontalFlip(0.5),
            T.RandomVerticalFlip(0.5),
            T.ColorJitter(brightness=0.15, contrast=0.15,
                          saturation=0.12, hue=0.008),
            T.RandomApply([T.GaussianBlur(3, sigma=(0.1, 1.0))], p=0.2),
            T.RandomApply([GaussianNoise(0.02)], p=0.15),
            T.RandomAffine(degrees=0, translate=(0.05, 0.05),
                           scale=(0.95, 1.05)),
        ])

    def __call__(self, tensor_chw):
        return self.transform(tensor_chw)

augmentor = CystoAugmentor()
print("✓ Augmentor ready")
print(f"  Augmentation strategy:")
print(f"    - Rotation: 360°")
print(f"    - Horizontal/Vertical Flip: 50%")
print(f"    - Color Jitter: brightness=0.15, contrast=0.15, saturation=0.12, hue=0.008")
print(f"    - Gaussian Blur: 20% probability")
print(f"    - Gaussian Noise: 15% probability")
print(f"    - Affine Transform: translate=0.05, scale=0.95-1.05")
print(f"  Each patch generates {1 + CFG.n_augments} versions (1 original + {CFG.n_augments} augmented)")

In [ ]:
# ============================================================
# CELL 7: DUAL BACKBONE — DINOv2 + DenseNet121
# ============================================================

def load_dinov2():
    """Try loading larger DINOv2 models first for better features."""
    print("  Loading DINOv2...")
    
    # Try ViT-B/14 first (best performance)
    try:
        model = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitb14')
        feat_dim = 768
        print(f"  ✓ DINOv2 ViT-B/14 — dim={feat_dim}")
        model.eval()
        for p in model.parameters():
            p.requires_grad = False
        return model.to(DEVICE), feat_dim
    except Exception as e:
        print(f"  ⚠ DINOv2 ViT-B/14 failed: {e}")
    
    # Fallback to ViT-S/14
    try:
        model = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14')
        feat_dim = 384
        print(f"  ✓ DINOv2 ViT-S/14 — dim={feat_dim}")
        model.eval()
        for p in model.parameters():
            p.requires_grad = False
        return model.to(DEVICE), feat_dim
    except Exception as e:
        print(f"  ⚠ DINOv2 ViT-S/14 failed: {e}")
    
    # Final fallback to original DINO
    try:
        model = torch.hub.load('facebookresearch/dino:main', 'dino_vits16')
        feat_dim = 384
        print(f"  ✓ DINO ViT-S/16 fallback — dim={feat_dim}")
        model.eval()
        for p in model.parameters():
            p.requires_grad = False
        return model.to(DEVICE), feat_dim
    except Exception as e2:
        print(f"  ⚠ All DINO models failed: {e2}")
        return None, 0


def load_densenet():
    model = models.densenet121(weights=models.DenseNet121_Weights.DEFAULT)
    feat_dim = model.classifier.in_features
    model.classifier = nn.Identity()
    model.eval()
    for p in model.parameters():
        p.requires_grad = False
    return model.to(DEVICE), feat_dim


print("Loading backbones...")
dino_model, dino_dim = load_dinov2()
dense_model, dense_dim = load_densenet()
print(f"  ✓ DenseNet121 — dim={dense_dim}")

if dino_model is not None:
    CFG.feat_dim = dino_dim + dense_dim
    backbone_desc = f"DINOv2({dino_dim}) + DenseNet({dense_dim})"
else:
    CFG.feat_dim = dense_dim
    backbone_desc = f"DenseNet({dense_dim})"

print(f"\n✓ Feature dim: {CFG.feat_dim} [{backbone_desc}]")

In [ ]:
# ============================================================
# CELL 8: FEATURE EXTRACTION — WITH AUGMENTATION
# ============================================================

def bgr_to_tensor(patch_bgr):
    rgb = cv2.cvtColor(patch_bgr, cv2.COLOR_BGR2RGB)
    return torch.from_numpy(rgb).permute(2, 0, 1).float() / 255.0


@torch.no_grad()
def extract_dual_features(tensor_list):
    """Extract concatenated features from both backbones."""
    all_feats = []
    bs = CFG.feat_batch
    for i in range(0, len(tensor_list), bs):
        batch = torch.stack(tensor_list[i:i+bs]).to(DEVICE)
        batch_norm = (batch - IMNET_MEAN) / IMNET_STD
        parts = []

        if dino_model is not None:
            dino_out = dino_model(batch_norm)
            if isinstance(dino_out, dict):
                dino_feats = dino_out.get('x_norm_clstoken', None)
                if dino_feats is None:
                    for v in dino_out.values():
                        if isinstance(v, torch.Tensor):
                            dino_feats = v; break
            else:
                dino_feats = dino_out
            if dino_feats.dim() > 2:
                dino_feats = dino_feats[:, 0, :]
            parts.append(dino_feats.cpu())

        dense_feats = dense_model(batch_norm)
        parts.append(dense_feats.cpu())

        all_feats.append(torch.cat(parts, dim=1))

    return torch.cat(all_feats, 0)


def extract_all_image_features():
    """Extract features WITH augmentation."""
    n_aug = CFG.n_augments
    print(f"\n{'='*55}")
    print(f"  Extracting features WITH augmentation")
    print(f"  Each patch → 1 orig + {n_aug} aug = {1+n_aug}x")
    print(f"  Max patches per image: {CFG.max_patches_per_image}")
    print(f"{'='*55}\n")

    image_data = []
    skipped = 0
    total_patches = 0

    for _, row in tqdm(df.iterrows(), total=len(df), desc="Images"):
        try:
            img = load_image(row.path, global_normalizer)
        except:
            skipped += 1
            continue

        patches, _, _ = extract_multiscale_patches(img)
        if len(patches) == 0:
            skipped += 1
            continue

        # Build tensors WITH augmentation
        tensors = []
        for p in patches:
            t = bgr_to_tensor(p)
            tensors.append(t)
            # Generate augmented versions
            for _ in range(n_aug):
                tensors.append(augmentor(bgr_to_tensor(p)))

        # Cap total tensors to avoid memory issues
        max_total = CFG.max_patches_per_image * (1 + n_aug)
        if len(tensors) > max_total:
            indices = random.sample(range(len(tensors)), max_total)
            tensors = [tensors[i] for i in sorted(indices)]

        feats = extract_dual_features(tensors)

        image_data.append({
            'features': feats,
            'label': row.target,
            'label_name': row.label,
            'patient': row.patient,
            'path': row.path,
            'n_patches': feats.shape[0]
        })
        total_patches += feats.shape[0]

    mem_bytes = sum(d['features'].element_size() * d['features'].nelement()
                    for d in image_data)

    print(f"\n✓ WITH augmentation:")
    print(f"  Images: {len(image_data)} ({skipped} skipped)")
    print(f"  Total patches: {total_patches}")
    print(f"  Memory: {mem_bytes/1e9:.2f} GB")

    return image_data


# ---- Extract features ----
t0 = time.time()
image_data = extract_all_image_features()
t1 = time.time()
print(f"Time: {(t1-t0)/60:.1f} min\n")

# Free backbones
del dino_model, dense_model
torch.cuda.empty_cache(); gc.collect()
print("✓ Backbones freed from GPU")

In [ ]:
# ============================================================
# CELL 9: CLAM WITH CLASS-WEIGHTED LOSS
# ============================================================

class CLAM(nn.Module):
    def __init__(self, feat_dim=CFG.feat_dim, hidden=CFG.mil_hidden,
                 n_classes=CFG.num_classes, dropout=CFG.mil_dropout,
                 k_sample=CFG.clam_k_sample):
        super().__init__()
        self.n_classes = n_classes
        self.k_sample = k_sample
        self.feat_noise = CFG.feat_noise_std
        self.feat_drop = nn.Dropout(CFG.feat_drop_p)

        self.fc = nn.Sequential(
            nn.Linear(feat_dim, hidden),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        self.att_net = nn.Sequential(
            nn.Linear(hidden, hidden // 2), nn.Tanh()
        )
        self.gate_net = nn.Sequential(
            nn.Linear(hidden, hidden // 2), nn.Sigmoid()
        )
        self.att_branches = nn.ModuleList([
            nn.Linear(hidden // 2, 1) for _ in range(n_classes)
        ])
        self.inst_classifiers = nn.ModuleList([
            nn.Sequential(
                nn.Linear(hidden, 64), nn.ReLU(), nn.Linear(64, 2)
            ) for _ in range(n_classes)
        ])
        self.bag_classifiers = nn.ModuleList([
            nn.Linear(hidden, 1) for _ in range(n_classes)
        ])

    def _inst_loss(self, scores, h, classifier, k):
        N = scores.shape[0]
        k = min(k, N // 2, 8)
        if k < 1:
            return torch.tensor(0.0, device=h.device)
        top_idx = torch.topk(scores, k).indices
        bot_idx = torch.topk(scores, k, largest=False).indices
        feats = torch.cat([h[top_idx], h[bot_idx]], dim=0)
        labels = torch.cat([
            torch.ones(k, dtype=torch.long),
            torch.zeros(k, dtype=torch.long)
        ]).to(h.device)
        return F.cross_entropy(classifier(feats), labels)

    def forward(self, x, label=None):
        if self.training:
            x = x + torch.randn_like(x) * self.feat_noise
            x = self.feat_drop(x)

        h = self.fc(x)
        att = self.att_net(h) * self.gate_net(h)

        logits = []
        total_inst = torch.tensor(0.0, device=x.device)

        for c in range(self.n_classes):
            a_scores = self.att_branches[c](att).squeeze(-1)
            a_weights = F.softmax(a_scores, dim=0)
            bag = torch.sum(a_weights.unsqueeze(-1) * h, dim=0)
            logits.append(self.bag_classifiers[c](bag))

            if self.training and label is not None and label.item() == c:
                total_inst += self._inst_loss(
                    a_scores.detach(), h,
                    self.inst_classifiers[c], self.k_sample
                )

        return {
            'logits': torch.cat(logits),
            'inst_loss': total_inst
        }


def compute_loss(output, label, class_weights=None):
    """Class-weighted bag loss + instance clustering loss."""
    bag_loss = F.cross_entropy(
        output['logits'].unsqueeze(0),
        label.unsqueeze(0),
        weight=class_weights
    )
    return CFG.bag_loss_w * bag_loss + CFG.inst_loss_w * output['inst_loss']


# Verify
m = CLAM().to(DEVICE)
x = torch.randn(30, CFG.feat_dim).to(DEVICE)
l = torch.tensor(2, dtype=torch.long).to(DEVICE)
m.train()
o = m(x, label=l)
print(f"✓ CLAM: (30, {CFG.feat_dim}) → logits {o['logits'].shape}")
n_p = sum(p.numel() for p in m.parameters() if p.requires_grad)
print(f"  Params: {n_p:,}")
del m, x, o; torch.cuda.empty_cache()

In [ ]:
# ============================================================
# CELL 10: TRAINING WITH EARLY STOPPING + PREDICTION
# ============================================================

def train_clam(model, train_images, val_images=None, epochs=CFG.epochs,
               class_weights=None, verbose=False):
    """Train CLAM with optional early stopping."""
    model.train()
    optimizer = torch.optim.Adam(model.parameters(), lr=CFG.lr, weight_decay=CFG.wd)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    best_val_loss = float('inf')
    best_state = None
    patience_counter = 0

    for epoch in range(epochs):
        # Training
        model.train()
        random.shuffle(train_images)
        epoch_loss = 0
        n_samples = 0

        for img_data in train_images:
            feats = img_data['features'].to(DEVICE)
            lbl = torch.tensor(img_data['label'], dtype=torch.long).to(DEVICE)

            if feats.shape[0] > CFG.max_patches_train:
                idx = torch.randperm(feats.shape[0])[:CFG.max_patches_train]
                feats = feats[idx]

            out = model(feats, label=lbl)
            loss = compute_loss(out, lbl, class_weights)

            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), CFG.grad_clip)
            optimizer.step()

            epoch_loss += loss.item()
            n_samples += 1

        scheduler.step()
        avg_train = epoch_loss / max(n_samples, 1)

        # Validation
        if val_images is not None and len(val_images) > 0:
            model.eval()
            val_loss = 0
            with torch.no_grad():
                for img_data in val_images:
                    feats = img_data['features'].to(DEVICE)
                    lbl = torch.tensor(img_data['label'], dtype=torch.long).to(DEVICE)
                    if feats.shape[0] > CFG.max_patches_test:
                        idx = torch.randperm(feats.shape[0])[:CFG.max_patches_test]
                        feats = feats[idx]
                    out = model(feats, label=lbl)
                    val_loss += compute_loss(out, lbl, class_weights).item()
            avg_val = val_loss / max(len(val_images), 1)

            if avg_val < best_val_loss:
                best_val_loss = avg_val
                best_state = copy.deepcopy(model.state_dict())
                patience_counter = 0
            else:
                patience_counter += 1

            if patience_counter >= CFG.patience:
                if verbose:
                    print(f"    Early stop at epoch {epoch+1}")
                break

            if verbose and (epoch + 1) % 20 == 0:
                print(f"    Epoch {epoch+1}/{epochs} "
                      f"train={avg_train:.4f} val={avg_val:.4f}")
        else:
            if verbose and (epoch + 1) % 20 == 0:
                print(f"    Epoch {epoch+1}/{epochs} train={avg_train:.4f}")

    if best_state is not None:
        model.load_state_dict(best_state)

    return model


@torch.no_grad()
def predict_image(model, feats):
    model.eval()
    feats = feats.to(DEVICE)
    if feats.shape[0] > CFG.max_patches_test:
        idx = torch.randperm(feats.shape[0])[:CFG.max_patches_test]
        feats = feats[idx]
    out = model(feats)
    probs = F.softmax(out['logits'], dim=0)
    return probs.argmax().item(), probs.max().item(), probs.cpu().numpy()


def predict_image_ensemble(models_list, feats, use_tta=False):
    """Ensemble prediction with optional Test-Time Augmentation."""
    all_probs = []
    
    if use_tta and CFG.use_tta:
        for model in models_list:
            tta_probs = []
            for _ in range(CFG.tta_rounds):
                _, _, probs = predict_image(model, feats)
                tta_probs.append(probs)
            model_avg_probs = np.mean(tta_probs, axis=0)
            all_probs.append(model_avg_probs)
    else:
        for model in models_list:
            _, _, probs = predict_image(model, feats)
            all_probs.append(probs)
    
    avg = np.mean(all_probs, axis=0)
    return np.argmax(avg), avg.max(), avg


print("✓ Train/predict ready (with TTA support)")

In [ ]:
# ============================================================
# CELL 11: LOPO ENGINE
# ============================================================

def compute_fold_class_weights(train_images):
    """Compute class weights for this specific fold's training data."""
    counts = Counter(d['label'] for d in train_images)
    total = sum(counts.values())
    weights = torch.zeros(CFG.num_classes)
    for c in range(CFG.num_classes):
        count = counts.get(c, 0)
        weights[c] = total / (CFG.num_classes * max(count, 1))
    return weights.to(DEVICE)


def create_validation_split(train_images, val_fraction=0.15):
    """Split training images into train/val BY PATIENT."""
    patient_groups = defaultdict(list)
    for d in train_images:
        patient_groups[d['patient']].append(d)

    patients = list(patient_groups.keys())
    if len(patients) < 3:
        return train_images, []

    # Use 2 patients for validation if we have 8+ patients
    n_val_patients = 2 if len(patients) >= 8 else 1
    
    patient_sizes = [(p, len(imgs)) for p, imgs in patient_groups.items()]
    patient_sizes.sort(key=lambda x: x[1])
    
    mid_idx = len(patient_sizes) // 2
    if n_val_patients == 2:
        val_pids = [patient_sizes[mid_idx-1][0], patient_sizes[mid_idx][0]]
    else:
        val_pids = [patient_sizes[mid_idx][0]]

    val_imgs = []
    for p in val_pids:
        val_imgs.extend(patient_groups[p])
    
    tr_imgs = [d for d in train_images if d['patient'] not in val_pids]

    return tr_imgs, val_imgs


def run_lopo(image_data, tag="WITH AUGMENTATION"):
    """Complete LOPO evaluation."""
    print(f"\n{'='*60}")
    print(f"  LOPO: {tag}")
    print(f"  {N_PATIENTS} folds × {CFG.n_ensemble} seeds")
    print(f"  Epochs: {CFG.epochs} (early stop patience={CFG.patience})")
    print(f"{'='*60}\n")

    fold_results = []
    all_y_true = []
    all_y_pred = []

    for fold_idx, test_pid in enumerate(PATIENTS):
        fold_start = time.time()
        print(f"\nFold {fold_idx+1}/{N_PATIENTS} — Test: P{test_pid}")

        train_images = [d for d in image_data if d['patient'] != test_pid]
        test_images = [d for d in image_data if d['patient'] == test_pid]

        if len(test_images) == 0:
            print(f"  ⚠ No test images, skipping")
            continue

        fold_weights = compute_fold_class_weights(train_images)
        tr_imgs, val_imgs = create_validation_split(train_images)

        test_dist = Counter(d['label_name'] for d in test_images)
        print(f"  Train: {len(tr_imgs)} imgs, Val: {len(val_imgs)} imgs, "
              f"Test: {len(test_images)} imgs")
        print(f"  Test dist: {dict(test_dist)}")

        # Train ensemble
        models_list = []
        for seed_off in range(CFG.n_ensemble):
            seed_everything(SEED + fold_idx * 100 + seed_off)
            model = CLAM().to(DEVICE)
            model = train_clam(
                model, tr_imgs,
                val_images=val_imgs,
                epochs=CFG.epochs,
                class_weights=fold_weights,
                verbose=(seed_off == 0)
            )
            models_list.append(model)

        # Predict with TTA
        fold_true = []
        fold_pred = []
        fold_conf = []

        for img_data in test_images:
            pred, conf, probs = predict_image_ensemble(
                models_list, img_data['features'], use_tta=True
            )
            fold_true.append(img_data['label'])
            fold_pred.append(pred)
            fold_conf.append(conf)

        for m in models_list:
            del m
        torch.cuda.empty_cache()

        fold_true = np.array(fold_true)
        fold_pred = np.array(fold_pred)
        fold_acc = accuracy_score(fold_true, fold_pred)
        fold_bal = balanced_accuracy_score(fold_true, fold_pred)
        fold_time = time.time() - fold_start

        print(f"  Accuracy:  {fold_acc:.4f} ({(fold_true==fold_pred).sum()}/{len(fold_true)})")
        print(f"  Balanced:  {fold_bal:.4f}")
        print(f"  Time:      {fold_time/60:.1f} min")

        fold_results.append({
            'patient': test_pid,
            'n_test_images': len(test_images),
            'image_accuracy': fold_acc,
            'balanced_accuracy': fold_bal,
            'correct': int((fold_true == fold_pred).sum()),
            'total': len(fold_true),
            'distribution': dict(test_dist),
            'mean_confidence': float(np.mean(fold_conf)),
            'per_image_true': fold_true.tolist(),
            'per_image_pred': fold_pred.tolist()
        })

        all_y_true.extend(fold_true.tolist())
        all_y_pred.extend(fold_pred.tolist())

    all_y_true = np.array(all_y_true)
    all_y_pred = np.array(all_y_pred)

    patient_accs = [f['image_accuracy'] for f in fold_results]
    patient_bals = [f['balanced_accuracy'] for f in fold_results]

    results = {
        'tag': tag,
        'folds': fold_results,
        'mean_patient_accuracy': float(np.mean(patient_accs)),
        'std_patient_accuracy': float(np.std(patient_accs)),
        'mean_patient_balanced': float(np.mean(patient_bals)),
        'std_patient_balanced': float(np.std(patient_bals)),
        'overall_image_accuracy': float(accuracy_score(all_y_true, all_y_pred)),
        'overall_balanced_accuracy': float(balanced_accuracy_score(all_y_true, all_y_pred)),
        'overall_confusion': confusion_matrix(
            all_y_true, all_y_pred,
            labels=list(range(CFG.num_classes))
        ).tolist(),
        'total_correct': int((all_y_true == all_y_pred).sum()),
        'total_images': len(all_y_true),
        'all_y_true': all_y_true.tolist(),
        'all_y_pred': all_y_pred.tolist()
    }

    print(f"\n{'─'*60}")
    print(f"  {tag}")
    print(f"  Mean patient acc:     {results['mean_patient_accuracy']:.4f} "
          f"± {results['std_patient_accuracy']:.4f}")
    print(f"  Mean patient bal:     {results['mean_patient_balanced']:.4f} "
          f"± {results['std_patient_balanced']:.4f}")
    print(f"  Overall image acc:    {results['overall_image_accuracy']:.4f}")
    print(f"  Overall balanced:     {results['overall_balanced_accuracy']:.4f}")
    print(f"  Total: {results['total_correct']}/{results['total_images']}")
    print(f"{'─'*60}")
    print(classification_report(
        all_y_true, all_y_pred,
        target_names=CFG.class_names,
        zero_division=0
    ))

    return results

print("✓ LOPO engine ready")

In [ ]:
# ============================================================
# CELL 12: RUN EXPERIMENT
# ============================================================

total_start = time.time()

print("=" * 60)
print("  EXPERIMENT: WITH AUGMENTATION")
print("=" * 60)

results = run_lopo(image_data, tag="WITH AUGMENTATION")

total_time = time.time() - total_start
print(f"\nTotal runtime: {total_time/60:.1f} min")

In [ ]:
# ============================================================
# CELL 13: VISUALIZATION
# ============================================================

fig = plt.figure(figsize=(18, 12))
gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.3, wspace=0.3)

# Per-patient accuracy
ax1 = fig.add_subplot(gs[0, :])
pids = [f"P{f['patient']}" for f in results['folds']]
accs = [f['image_accuracy'] for f in results['folds']]

x = np.arange(len(pids))
bars = ax1.bar(x, accs, color='#4CAF50', alpha=0.8)
ax1.set_ylabel('Image Accuracy')
ax1.set_title(f'Per-Patient Image Accuracy (Mean: {results["mean_patient_accuracy"]:.3f} ± {results["std_patient_accuracy"]:.3f})',
              fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels(pids, rotation=45)
ax1.set_ylim(0, 1.1)
ax1.grid(axis='y', alpha=0.3)
ax1.axhline(results['mean_patient_accuracy'], color='red', linestyle='--', linewidth=2, label='Mean')
ax1.legend()

# Confusion matrix
ax2 = fig.add_subplot(gs[1, 0])
cm = np.array(results['overall_confusion'])
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=CFG.class_names, yticklabels=CFG.class_names, ax=ax2)
ax2.set_xlabel('Predicted')
ax2.set_ylabel('True')
ax2.set_title(f'Confusion Matrix\nOverall Acc={results["overall_image_accuracy"]:.3f}',
              fontweight='bold')

# Per-class recall
ax3 = fig.add_subplot(gs[1, 1])
recalls = []
for i in range(CFG.num_classes):
    total = cm[i].sum()
    recalls.append(cm[i, i] / max(total, 1))

y_pos = np.arange(CFG.num_classes)
bars = ax3.barh(y_pos, recalls, color='#4CAF50', alpha=0.8)
for i in range(CFG.num_classes):
    ax3.text(recalls[i] + 0.02, i, f'{recalls[i]:.3f}', va='center', fontsize=10)
ax3.set_yticks(y_pos)
ax3.set_yticklabels(CFG.class_names)
ax3.set_xlabel('Recall')
ax3.set_title('Per-Class Recall', fontweight='bold')
ax3.set_xlim(0, 1.2)
ax3.grid(axis='x', alpha=0.3)

fig.suptitle(
    f'LOPO Classification WITH Augmentation\n'
    f'Pipeline: Lab Norm → Multi-Scale Patches → Augmentation → {backbone_desc} → CLAM',
    fontsize=14, fontweight='bold', y=1.00
)
plt.savefig('lopo_with_aug_results.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Saved lopo_with_aug_results.png")

In [ ]:
# ============================================================
# CELL 14: EXPORT RESULTS
# ============================================================

summary = {
    'pipeline': f'Lab_Norm + MultiScale + Augmentation + {backbone_desc} + CLAM',
    'evaluation': 'LOPO',
    'augmentation': True,
    'classes': CFG.class_names,
    'n_patients': N_PATIENTS,
    'total_images': len(df),
    'results': {
        'mean_patient_acc': results['mean_patient_accuracy'],
        'std_patient_acc': results['std_patient_accuracy'],
        'mean_patient_bal': results['mean_patient_balanced'],
        'std_patient_bal': results['std_patient_balanced'],
        'overall_image_acc': results['overall_image_accuracy'],
        'overall_balanced_acc': results['overall_balanced_accuracy'],
        'total_correct': results['total_correct'],
        'total_images': results['total_images'],
        'confusion_matrix': results['overall_confusion'],
        'folds': results['folds']
    },
    'config': {
        'patch_scales': CFG.patch_scales,
        'max_patches_per_image': CFG.max_patches_per_image,
        'feature_dim': CFG.feat_dim,
        'backbone': backbone_desc,
        'mil_model': 'CLAM',
        'clam_k_sample': CFG.clam_k_sample,
        'epochs': CFG.epochs,
        'early_stopping_patience': CFG.patience,
        'lr': CFG.lr,
        'n_ensemble': CFG.n_ensemble,
        'n_augments': CFG.n_augments,
        'class_weighted_loss': True,
        'use_tta': CFG.use_tta,
        'tta_rounds': CFG.tta_rounds,
        'augmentations': [
            'rotation_360', 'hflip', 'vflip',
            'color_jitter(b=0.15,c=0.15,s=0.12,h=0.008)',
            'gaussian_blur(p=0.2)',
            'gaussian_noise(p=0.15)',
            'affine(t=0.05,s=0.95-1.05)'
        ],
        'total_runtime_minutes': total_time / 60
    }
}

with open('lopo_results_with_aug.json', 'w') as f:
    json.dump(summary, f, indent=2, default=str)
print("✓ Saved lopo_results_with_aug.json")

# Per-patient CSV
rows = []
for f in results['folds']:
    rows.append({
        'patient': f['patient'],
        'n_test_images': f['n_test_images'],
        'accuracy': round(f['image_accuracy'], 4),
        'balanced_accuracy': round(f['balanced_accuracy'], 4),
        'confidence': round(f['mean_confidence'], 4)
    })
pd.DataFrame(rows).to_csv('lopo_per_patient_with_aug.csv', index=False)
print("✓ Saved lopo_per_patient_with_aug.csv")

# Classification report
with open('classification_report_with_aug.txt', 'w') as f:
    f.write(f"WITH AUGMENTATION\n{'='*50}\n\n")
    f.write(classification_report(
        results['all_y_true'], results['all_y_pred'],
        target_names=CFG.class_names, zero_division=0
    ))
print("✓ Saved classification_report_with_aug.txt")

print(f"\n{'='*60}")
print("  EXPERIMENT COMPLETE - WITH AUGMENTATION")
print(f"{'='*60}")
print(f"\nFinal Results:")
print(f"  Mean Patient Accuracy: {results['mean_patient_accuracy']:.4f} ± {results['std_patient_accuracy']:.4f}")
print(f"  Overall Image Accuracy: {results['overall_image_accuracy']:.4f}")
print(f"  Total Runtime: {total_time/60:.1f} minutes")
print(f"{'='*60}\n")
print("💡 Augmentation Strategy:")
print(f"  Each patch was augmented {CFG.n_augments}x with:")
print(f"  • 360° rotation, H/V flips")
print(f"  • Color jitter (brightness, contrast, saturation, hue)")
print(f"  • Gaussian blur & noise")
print(f"  • Affine transformations")
print(f"{'='*60}\n")